# Feature engineering

This notebook builds our engineered features and saves them to `data/processed/features_v1.parquet`, so a model
notebook can load them directly.

The functions are in `src/domain_features.py`. This notebook calls them, explains the reasoning behind each group
of features, and runs a quick check on the training block.

Rules we followed for every feature:

- Features that describe a customer's history only look at earlier transactions. The table is sorted by
  `TransactionDT` then `TransactionID`, so "earlier" means earlier in that order. This matches what a live system
  would know when a transaction comes in. `tests/test_domain_features.py` checks this.
- Anything learned from the data (the most common `addr2` and `card3` values, and the V-column missing groups)
  comes from the training block only, i.e. the first 80% of rows by time.
- No feature uses `isFraud`.
- Imputation, scaling and encoding are not done here. They belong in the model notebook and have to be fit on
  training rows only.

In [1]:
import sys; sys.path.insert(0, '..')
import time
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score

from src.data import load_joined
from src import domain_features as dfe

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

t0 = time.time()
df = load_joined()                       # time-sorted joined table from src/data.py
assert df['TransactionDT'].is_monotonic_increasing
train = df.iloc[:dfe.N_TRAIN]            # training block: first 80% by time
y_train = train['isFraud'].to_numpy()
print(df.shape, f'train block rows = {len(train):,}, train fraud rate = {y_train.mean():.4f}')

(590540, 436) train block rows = 472,432, train fraud rate = 0.0351


### Quick check

For each new feature we look at the training block only and report how much of it is missing, plus the AUROC
you get from that feature on its own (flipped if it's below 0.5, so 0.5 means no signal either way). For 0/1
flags we also show the fraud rate in each group.

This is just to see whether a feature behaves the way we expected. It isn't how we decide what to keep. A feature
can be weak on its own and still help the model, so that decision is made later with time-based CV.

In [2]:
def signal_table(frame):
    rows = []
    for c in frame.columns:
        s = frame[c].iloc[:dfe.N_TRAIN]
        row = {'feature': c, 'pct_missing': 100 * s.isna().mean()}
        if isinstance(s.dtype, pd.CategoricalDtype):
            row['n_levels'] = s.nunique()
        else:
            v = s.to_numpy(np.float64); m = ~np.isnan(v)
            if m.sum() and len(np.unique(y_train[m])) > 1 and len(np.unique(v[m])) > 1:
                auc = roc_auc_score(y_train[m], v[m]); row['auroc_1feat'] = max(auc, 1 - auc)
            if set(np.unique(v[m])) <= {0.0, 1.0}:
                row['fraud_rate_if_1'] = y_train[m][v[m] == 1].mean() if (v[m] == 1).any() else np.nan
                row['fraud_rate_if_0'] = y_train[m][v[m] == 0].mean() if (v[m] == 0).any() else np.nan
                row['pct_rows_1'] = 100 * (v[m] == 1).mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('feature').round(4)

def rate_by_level(frame, col, top=10):
    s = frame[col].iloc[:dfe.N_TRAIN].astype(object).fillna('<missing>')
    t = pd.DataFrame({'level': s, 'isFraud': y_train}).groupby('level')['isFraud'].agg(n='size', fraud_rate='mean')
    return t.sort_values('n', ascending=False).head(top).round(4)

## 0. Training-block constants and the customer ID

A couple of the flags below compare a value with the most common one, so we first need to know what's common.
We take the most common `addr2` (looks like a country code) and `card3` from the training block. We also take the
V-column groups from Step 2, i.e. columns that are missing on exactly the same rows.

Vesta marks a customer's later transactions as fraud once one of them is reported, so fraud tends to stick to a
customer rather than to a single transaction. The data has no customer ID, so following `Context.md` we
approximate one:

`uid = card1 _ addr1 _ floor(TransactionDay − D1)`

`card1` roughly identifies the card, `addr1` is the billing region, and `TransactionDay − D1` estimates the day
the account was opened, which stays the same for a given customer.

We use `uid` to group transactions when building the history features. It's saved in the output file so it can
be used for grouped CV or the returning vs new customer analysis, but it shouldn't be given to a model as a
feature, because the model would just memorise the IDs.

In [3]:
consts = dfe.fit_constants(train)
print({k: v for k, v in consts.items() if k != 'v_group_representatives'})
print('V-group representative columns:', consts['v_group_representatives'])

uid, fams = dfe.build_features(df, consts)
print(f'built {sum(f.shape[1] for f in fams.values())} features in {time.time() - t0:.1f}s')

sizes = uid.iloc[:dfe.N_TRAIN].value_counts()
sizes = sizes[sizes > 0]   # drop uid categories that only occur in later blocks
print(f"train-block uids: {len(sizes):,}; median group size {sizes.median():.0f}; "
      f"share of rows in single-transaction uids {100 * (sizes == 1).sum() / dfe.N_TRAIN:.1f}%")

{'addr2_mode': 87.0, 'card3_mode': 150.0, 'n_train_rows': 472432}
V-group representative columns: ['V279', 'V95', 'V281', 'V12', 'V53', 'V75', 'V35', 'V1', 'V220', 'V169', 'V167', 'V217', 'V322', 'V138']


built 73 features in 2.8s
train-block uids: 182,535; median group size 1; share of rows in single-transaction uids 22.5%


## 1. Transaction amount

| Feature | What it is | Why we think it helps |
| --- | --- | --- |
| `fe_amt_log` | log(1 + amount) | Amounts are very skewed. Trees don't mind, but LR and the MLP work better on a log scale. |
| `fe_amt_cents` | the cents part (0 to 99) | Prices like 57.95 and 57.00 tend to come from different kinds of merchants and products. |
| `fe_amt_decimals` | number of decimal places (capped at 3) | Three or more decimals usually means the amount was converted from another currency, so the purchase is probably cross-border, which is riskier. |
| `fe_amt_round_dollar`, `fe_amt_round_10` | whole-dollar and multiple-of-10 flags | Fraudsters often buy round amounts such as gift cards or top-ups that are easy to resell. |
| `fe_amt_lt5` | amount under $5 | Stolen cards are often tested with a small purchase first to check they still work. |

These only use the transaction's own amount, so there's no leakage risk.

In [4]:
signal_table(fams['amount'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_amt_log,0.0,0.5019,NaN,NaN,NaN
fe_amt_cents,0.0,0.5606,NaN,NaN,NaN
fe_amt_decimals,0.0,0.5503,NaN,NaN,NaN
fe_amt_round_dollar,0.0,0.5075,0.0361,0.0341,51.8246
fe_amt_round_10,0.0,0.5191,0.0439,0.0336,14.7729
fe_amt_lt5,0.0,0.5009,0.0652,0.0351,0.1947


## 2. Time of day and customer habits

| Feature | What it is | Why we think it helps |
| --- | --- | --- |
| `fe_hour` | hour of day | We don't know the real start time of `TransactionDT`, so the hour is relative, but the daily pattern still shows up. Fraud is often more common at night when people aren't checking their accounts. |
| `fe_dow` | day of week (relative) | Normal shopping follows a weekly pattern. |
| `fe_uid_hour_dev` | how many hours this transaction is from the customer's usual time | Picks up a change in habit, e.g. someone who always shops in the evening suddenly buying something at 4am. It's measured around the clock, so 23:00 and 01:00 count as 2 hours apart. |
| `fe_uid_hour_concentration` | 0 to 1, how consistently the customer shops at the same time of day | Helps interpret the previous feature: an odd hour means more for someone with a regular routine. |

The customer's usual hour is worked out from their earlier transactions only, so it's missing for their first one.

In [5]:
signal_table(fams['time'])

,pct_missing,auroc_1feat
feature,,
fe_hour,0.0000,0.5093
fe_dow,0.0000,0.5132
fe_uid_hour_dev,38.6373,0.5245
fe_uid_hour_concentration,38.6373,0.5250


## 3. Normalised D columns

Several D columns count days since some event (D1 looks like days since the card was opened). These keep going
up over time for every customer, so the raw value is partly just standing in for the date. Subtracting them from
the transaction day gives a fixed date for each customer instead:

`fe_Dxn = TransactionDay − Dx` for D1, D4, D10 and D15

We expect this to help for two reasons:

- A fixed date doesn't drift over time, so a model trained on earlier months should carry over better to later
  ones. That matters because our split is by time.
- It works like a customer fingerprint (it's part of the uid). A known card showing up with a different
  account-open date could mean someone set up a new account with it.

Negative values just mean the account was opened before the data starts. Only the row's own values are used, so
there's no leakage.

In [6]:
signal_table(fams['d_norm'])

,pct_missing,auroc_1feat
feature,,
fe_D1n,0.1981,0.6272
fe_D4n,29.5861,0.6050
fe_D10n,14.0530,0.6277
fe_D15n,16.4606,0.6348


## 4. First-time use of a device, email, etc.

When an account is taken over, the fraudster usually has the card details but not the owner's phone, browser or
email address. So a customer suddenly using a device or email they've never used before is a useful signal.

| Feature | What it is |
| --- | --- |
| `fe_uid_new_{device, browser, os, screen, email, product}` | 1 if the customer has used other values before but never this one, 0 if they've used this one before. Missing if there's nothing to compare with (first transaction, or the value wasn't recorded). |
| `fe_uid_n_prior_{devices, emails, browsers}` | How many different values the customer has used so far. Lots of devices or emails on one card is unusual. |
| `fe_card1_new_addr1`, `fe_card1_n_prior_addr1`, `fe_card1_n_prior_emails` | The same idea using only the card (`card1`), mainly to catch a new billing address on a card we've already seen. |

We also look at it from the device side. Fraud rings often push many stolen cards through the same device, so we
build a rough device fingerprint from `DeviceInfo`, OS, browser and screen resolution, and count:

| Feature | What it is |
| --- | --- |
| `fe_fp_n_prior_cards` | different cards seen on this fingerprint so far |
| `fe_fp_n_prior_uids` | different customers seen on it so far |
| `fe_fp_prior_count` | number of earlier transactions from it |

Some `DeviceInfo` values are very generic (e.g. "Windows"), which is why we add OS, browser and resolution, but the
fingerprint is still fairly rough. These features are missing when there's no device information, which is the
case for most rows without an identity record.

All the counts in this section only use earlier rows.

In [7]:
signal_table(fams['novelty'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_uid_new_device,90.4361,0.5331,0.0711,0.0956,31.6380
fe_uid_new_browser,87.6966,0.5346,0.0785,0.1057,33.4262
fe_uid_new_os,95.2256,0.5955,0.0253,0.0676,34.6737
fe_uid_new_screen,95.6116,0.6005,0.0257,0.0722,35.2354
fe_uid_new_email,50.4481,0.5077,0.0373,0.0427,13.2175
fe_uid_new_product,38.6373,0.5066,0.0241,0.0408,3.1487
fe_uid_n_prior_devices,0.0000,0.5981,NaN,NaN,NaN
fe_uid_n_prior_emails,0.0000,0.5689,NaN,NaN,NaN
fe_uid_n_prior_browsers,0.0000,0.6107,NaN,NaN,NaN


## 5. Spending speed and escalation

Once a stolen card works, the fraudster usually tries to spend as much as possible before it gets blocked. That
shows up as bursts of transactions and growing amounts, so these features compare each transaction with the
customer's own history.

| Feature | Why we think it helps |
| --- | --- |
| `fe_uid_prior_count` | Number of earlier transactions. A customer with a long history is less likely to be fake. |
| `fe_uid_secs_since_prev` | Time since the customer's last transaction. Very short gaps suggest automated or rushed spending. |
| `fe_uid_count_{1h, 24h, 7d}` | Number of transactions in the last hour, day and week. |
| `fe_uid_spend_24h` | Amount spent in the previous 24 hours. |
| `fe_uid_spend_24h_rel_daily` | (spend in the last 24h + this amount) divided by the customer's average daily spend so far. Picks up a spending spree compared with their normal level. |
| `fe_uid_amt_rel_prev`, `_rel_prior_mean`, `_rel_prior_max` | This amount compared with the previous amount, the average and the largest so far. |
| `fe_uid_small_then_large` | The previous amount was under $5 and this one is at least 10 times bigger: test the card, then use it. |
| `fe_uid_repeat_amt_count` | How many times the customer has paid exactly this amount before. Repeated amounts are typical of subscriptions and bills, which are rarely fraud. |

The time windows only count transactions before the current one (if two share a timestamp, only the one earlier in
the table counts). Some of these overlap with the F1/F2 aggregates in the group plan; we kept them so this table
can be used on its own.

One feature we left out on purpose is "was the customer's previous transaction fraud?". It would score very well
because of how Vesta labels customers, but it isn't realistic: chargebacks come in weeks later, so a real system
wouldn't know that label yet. It's also basically target encoding of `uid`, which `Context.md` rules out.

In [8]:
signal_table(fams['velocity'])

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,
fe_uid_prior_count,0.0000,0.5738,NaN,NaN,NaN
fe_uid_secs_since_prev,38.6373,0.6732,NaN,NaN,NaN
fe_uid_amt_rel_prev,38.6373,0.5129,NaN,NaN,NaN
fe_uid_amt_rel_prior_mean,38.6373,0.5167,NaN,NaN,NaN
fe_uid_amt_rel_prior_max,38.6373,0.5051,NaN,NaN,NaN
fe_uid_small_then_large,38.6373,0.5004,0.1061,0.0403,0.0455
fe_uid_repeat_amt_count,0.0000,0.5463,NaN,NaN,NaN
fe_uid_count_1h,0.0000,0.5788,NaN,NaN,NaN
fe_uid_count_24h,0.0000,0.6231,NaN,NaN,NaN


## 6. Email, browser and device details

| Feature | Why we think it helps |
| --- | --- |
| `fe_P_email_provider`, `fe_R_email_provider` | The ~60 email domains grouped into providers (google, yahoo, microsoft, apple, aol, anonymous, other). A few bigger groups are easier to learn from than lots of small domains. |
| `fe_P_email_tld` | The end of the domain (com, net, mx, es, ...). A foreign ending suggests a buyer from another country. |
| `fe_email_anonymous` | Anonymous or privacy-focused email services are easy to sign up for and hide who's behind them. |
| `fe_email_match` | Whether the purchaser and recipient domains match. A mismatch could be a gift, but it could also be goods sent to someone else. Missing if either one is blank. |
| `fe_browser_name`, `fe_browser_generic` | Browser family from `id_31`. "Generic" browser strings can point to bots or faked user agents. |
| `fe_browser_versions_behind` | How far behind this browser version is compared with the newest version of the same browser seen so far. Fraud setups sometimes use old or emulated browsers. |
| `fe_os_family`, `fe_device_brand` | OS family from `id_30` and device brand from `DeviceInfo` (Samsung "SM-..." codes, Motorola, Huawei, Apple, Windows, ...). This turns hundreds of model strings into a few groups. |
| `fe_screen_w`, `_h`, `_area`, `_aspect` | Taken from `id_33` (e.g. "1920x1080"). Emulators and virtual machines often have unusual screen sizes. "0x0" is treated as missing. |
| `fe_devicetype_os_mismatch` | The device type says desktop but the OS is iOS or Android, or the other way round. A setup that doesn't add up could mean spoofing. |

Most of these just parse the row's own values. `fe_browser_versions_behind` uses a running maximum over earlier rows.

In [9]:
display(signal_table(fams['identity']))
rate_by_level(fams['identity'], 'fe_P_email_provider')

,pct_missing,n_levels,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1
feature,,,,,,
fe_P_email_provider,15.5760,7.0,NaN,NaN,NaN,NaN
fe_R_email_provider,75.9500,7.0,NaN,NaN,NaN,NaN
fe_P_email_tld,15.5760,9.0,NaN,NaN,NaN,NaN
fe_email_anonymous,15.5760,NaN,0.5128,0.0242,0.0371,7.5287
fe_email_match,77.5695,NaN,0.5672,0.0929,0.0282,81.0124
fe_browser_name,75.1365,42.0,NaN,NaN,NaN,NaN
fe_browser_generic,75.1365,NaN,0.5165,0.0894,0.0739,18.4731
fe_browser_versions_behind,80.0797,NaN,0.5005,NaN,NaN,NaN
fe_os_family,86.0109,6.0,NaN,NaN,NaN,NaN


,n,fraud_rate
level,,
google,183103,0.0439
yahoo,84899,0.0226
<missing>,73586,0.0298
microsoft,48667,0.0519
anonymous,30028,0.0242
aol,22886,0.0234
other,22603,0.0204
apple,6660,0.0279


## 7. Location and card type

| Feature | Why we think it helps |
| --- | --- |
| `fe_addr2_not_mode` | `addr2` looks like a country code, and one value covers about 99% of training rows. Any other value most likely means a foreign billing address. |
| `fe_card3_not_mode` | `card3` has one value for about 88% of training rows. The other values may be cards issued abroad, though that's a guess since the column is masked. |
| `fe_card4_card6` | Card network and card type together (e.g. visa_debit, discover_credit), since fraud rates differ by the combination. |
| `fe_uid_dist1_rel_prior_mean` | `dist1` (a masked distance, possibly billing to shipping) compared with the customer's past average. Shipping much further than usual can be a warning sign. |

The most common values come from the training block, and the distance ratio only uses earlier rows.

In [10]:
display(signal_table(fams['geo_card']))
rate_by_level(fams['geo_card'], 'fe_card4_card6')

,pct_missing,auroc_1feat,fraud_rate_if_1,fraud_rate_if_0,pct_rows_1,n_levels
feature,,,,,,
fe_addr2_not_mode,11.3796,0.5152,0.1038,0.0243,0.9389,NaN
fe_card3_not_mode,0.1742,0.6334,0.1116,0.0249,11.8259,NaN
fe_card4_card6,0.0000,NaN,NaN,NaN,NaN,15.0
fe_uid_dist1_rel_prior_mean,78.8522,0.5202,NaN,NaN,NaN,NaN


,n,fraud_rate
level,,
visa_debit,238826,0.0254
mastercard_debit,110604,0.0209
visa_credit,69303,0.0670
mastercard_credit,40228,0.0734
american express_credit,7077,0.0285
discover_credit,5106,0.0758
nan_nan,823,0.0292
discover_debit,288,0.0486
american express_debit,117,0.0342


## 8. Missing values and the M and C columns

In this dataset, whether a value is filled in tells you something. Identity data only exists for some merchants
and products, and whole blocks of V columns are present or missing depending on the type of transaction.

| Feature | Why we think it helps |
| --- | --- |
| `fe_n_missing` | Number of missing raw columns in the row, a rough summary of what data the merchant sent. |
| `fe_v_missing_pattern` | Which of the 14 V-column groups are missing (one column checked per group), stored as a pattern ID. Different patterns roughly line up with different products or checkout flows. |
| `fe_m_true`, `fe_m_false`, `fe_m_missing` | The M columns are match checks (e.g. does the name or address match the card). We count how many matched, didn't match, or weren't checked. M4 has different values (M0/M1/M2), so it's left as it is. |
| `fe_uid_c1_rel_prior_mean`, `fe_uid_c13_rel_prior_mean` | The C columns count things linked to the card, like addresses or devices. A big jump compared with the customer's own past could mean a lot of new links. We picked C1 and C13 as examples; other C columns could be added the same way. |

These are counts from the row itself, plus past-only averages for the C ratios.

In [11]:
signal_table(fams['missing'])

,pct_missing,auroc_1feat,n_levels
feature,,,
fe_n_missing,0.0000,0.6652,NaN
fe_v_missing_pattern,0.0000,NaN,209.0
fe_m_true,0.0000,0.6386,NaN
fe_m_false,0.0000,0.6091,NaN
fe_m_missing,0.0000,0.6385,NaN
fe_uid_c1_rel_prior_mean,38.6523,0.6049,NaN
fe_uid_c13_rel_prior_mean,41.6653,0.5340,NaN


## 9. How common a value is

For card, address, email domain, device, browser and screen size, we take the number of earlier rows with the
same value divided by the number of earlier rows. Rare devices, browsers and email domains tend to be riskier, and
a card or address that suddenly shows up a lot is worth flagging.

Computing these over the whole table would use future rows, so we only count earlier ones. The values are a bit
noisy at the very start of the data, when there aren't many earlier rows yet.

In [12]:
signal_table(fams['frequency'])

,pct_missing,auroc_1feat
feature,,
fe_freq_card1,0.0002,0.5168
fe_freq_addr1,11.3798,0.5279
fe_freq_P_emaildomain,15.5760,0.5532
fe_freq_DeviceInfo,78.9311,0.5888
fe_freq_id_31,75.1365,0.5842
fe_freq_id_33,86.9160,0.5641


## 10. Putting it together and saving

The output has every raw column from the joined table, `uid`, and all the `fe_` columns, in the original time
order. The splits are based on row position, so the table must not be re-sorted. Numbers are stored as `float32`
and text as `category`, the same as the rest of the repo.

We also write the list of columns to `config/features_v1.yaml`, so the model notebook can pick features without
typing out the names.

In [13]:
full = dfe.assemble(df, uid, fams)
new_cols = [c for f in fams.values() for c in f.columns]

assert len(full) == len(df) and (full['TransactionID'].to_numpy() == df['TransactionID'].to_numpy()).all()
assert full['TransactionDT'].is_monotonic_increasing
assert full.columns.is_unique
assert not any(full[c].dtype == object for c in full.columns)
print(f'{full.shape[0]:,} rows x {full.shape[1]} columns ({len(new_cols)} new); '
      f'{full.memory_usage(deep=True).sum() / 1024**3:.2f} GB in memory')

590,540 rows x 510 columns (73 new); 1.07 GB in memory


In [14]:
raw_feature_cols = [c for c in df.columns if c not in dfe.ID_COLS and c != dfe.TARGET]
feature_list = {
    'version': 'features_v1',
    'parquet': str(dfe.FEATURES_PATH.relative_to(dfe.ROOT)),
    'built_by': 'notebooks/feature_engineering.ipynb (logic in src/domain_features.py)',
    'target': dfe.TARGET,
    'not_features': {
        'TransactionID': 'row identifier',
        'TransactionDT': 'raw time; trends with time, so a model would extrapolate poorly on later blocks',
        'TransactionDay': 'same as TransactionDT',
        'uid': 'customer identifier for grouping and returning-vs-new analysis; raw ID would be memorised',
    },
    'train_block_rows': dfe.N_TRAIN,
    'constants_fit_on_train_block': consts,
    'raw_feature_cols': raw_feature_cols,
    'engineered_cols': {name: list(f.columns) for name, f in fams.items()},
    'categorical_cols': [c for c in raw_feature_cols + new_cols
                         if isinstance(full[c].dtype, pd.CategoricalDtype)],
    'notes': [
        'V columns: all 339 are included; the pruning list in config/features.yaml is still awaiting group approval.',
        'Fit imputers, scalers and encoders in the model notebook on training rows only (inside each CV fold).',
        'Test block (last 10% of rows) stays unscored until the experiment freeze.',
    ],
}

dfe.FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
full.to_parquet(dfe.FEATURES_PATH, index=False)
with open(dfe.FEATURE_LIST_PATH, 'w') as f:
    yaml.safe_dump(feature_list, f, sort_keys=False, width=110)

print('saved', dfe.FEATURES_PATH.relative_to(dfe.ROOT),
      f'({dfe.FEATURES_PATH.stat().st_size / 1024**2:.0f} MB) and', dfe.FEATURE_LIST_PATH.relative_to(dfe.ROOT))

saved data/processed/features_v1.parquet (118 MB) and config/features_v1.yaml


## 11. All new features at a glance

Every new feature sorted by its single-feature AUROC on the training block. Same caveat as before: this is a
sanity check, not how we pick features.

In [15]:
overview = pd.concat({name: signal_table(f) for name, f in fams.items()}, names=['family', 'feature'])
overview.sort_values('auroc_1feat', ascending=False)

pct_missing  auroc_1feat  fraud_rate_if_1  fraud_rate_if_0  pct_rows_1  n_levels
family   feature                                                                                                     
velocity fe_uid_secs_since_prev          38.6373       0.6732              NaN              NaN         NaN       NaN
missing  fe_n_missing                     0.0000       0.6652              NaN              NaN         NaN       NaN
velocity fe_uid_spend_24h_rel_daily      38.6373       0.6548              NaN              NaN         NaN       NaN
         fe_uid_count_7d                  0.0000       0.6429              NaN              NaN         NaN       NaN
missing  fe_m_true                        0.0000       0.6386              NaN              NaN         NaN       NaN
...                                          ...          ...              ...              ...         ...       ...
identity fe_browser_name                 75.1365          NaN              NaN              NaN         NaN      42.0
         fe_os_family                    86.0109          NaN              NaN              NaN         NaN       6.0
         fe_device_brand                 78.9311          NaN              NaN              NaN         NaN      14.0
geo_card fe_card4_card6                   0.0000          NaN              NaN              NaN         NaN      15.0
missing  fe_v_missing_pattern             0.0000          NaN              NaN              NaN         NaN     209.0

[73 rows x 6 columns]

## Loading the file in a model notebook

```python
import pandas as pd, yaml
spec = yaml.safe_load(open('../config/features_v1.yaml'))
full = pd.read_parquet('../' + spec['parquet'])
features = spec['raw_feature_cols'] + [c for cols in spec['engineered_cols'].values() for c in cols]
X, y = full[features], full[spec['target']]
# split by row position: train = rows [0, 472432); validation and test follow (see Context.md)
```

Things to keep in mind:

- Split by row position. The table is already in time order, so don't shuffle it.
- Tune with time-based CV inside the training block, use the validation block for early stopping and choosing the
  threshold, and leave the test block alone until the freeze.
- XGBoost can use the category columns directly with `enable_categorical=True` and `tree_method='hist'`. Otherwise,
  convert them to integer codes.
- For LR and the MLP, fit imputation, scaling and encoding on the training rows only.
- Add feature groups one at a time, and keep a group if time-based CV PR-AUC goes up in most folds.
- Log every run to `results/log.csv` and note `features_v1` in the params.

Still to do: check each feature for consistency over time (train on the first month, score the last month of the
training block, and drop features that only work early on), and settle the V-column pruning once the group
approves the list.